# `02_population`: Population data for municipalities, provinces, and the H3 grid (Netherlands, 2025)

## Introduction

### Purpose

This notebook complements the spatial reference units produced in `01_boundaries` with 2025 population data at three levels. Municipality counts are retrieved from CBS (Table 86059NED), province totals are aggregated from municipalities, and H3 cell populations are sourced from the Kontur Population Dataset. The municipality table also carries the CBS degree-of-urbanisation classification (*stedelijkheid*), which the thesis uses to structure the urban–rural gradient analysis (§"Degree of urbanisation", Results).

### Inputs

- CBS Open Data API, Table 86059NED ("Areas in the Netherlands 2025"), reference date 1 January 2025.
- Kontur Population Dataset (Netherlands), snapshot 2023-11-01, distributed via the Humanitarian Data Exchange.

### Outputs

- `municipality_population`: one row per municipality, with `population`, the parent `province_code`/`province_name`, and `code_degree_of_urbanisation` / `degree_of_urbanisation` (Dutch CBS classification translated to English labels).
- `province_population`: one row per province, with `population` aggregated from municipalities.
- `h3_population`: one row per H3 cell at resolution 8, with `population`, `geom`, and a `resolution` validation column.

### Key steps

Municipality records are retrieved from the CBS Open Data API (Table 86059NED), with identifier and name columns renamed for downstream use and the Dutch degree-of-urbanisation classification translated to English labels. Province populations are computed by summing municipalities. H3-cell populations are loaded from the Kontur Population Dataset: string H3 indexes are converted to DuckDB's native H3 type, geometries are reprojected from EPSG:3857 to EPSG:4326, and the H3 resolution is attached as a validation column. Per the thesis (Table 3 and §3.4.6), municipality population feeds the per-capita extent metric (length per 1,000 inhabitants), while province and H3 population enter only via the MAUP robustness check in stage 08.

### Dependencies on prior notebooks

None at runtime; this notebook does not `%run` any prior notebook. Conceptually, the H3 grid generated in `01_boundaries` is the spatial counterpart of `h3_population` here, and both use H3 resolution 8.

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Municipality](#2-municipality)
3. [Province](#3-province)
4. [H3 grid](#4-h3-grid)

---

## 1. Environment setup

### Libraries and extensions

In [4]:
import duckdb
import requests
import pandas as pd
import geopandas as gpd

### Install DuckDB extensions

Load the spatial extension (vector geometry support) and the H3 community extension (native H3 type and helper functions used to validate the Kontur resolution).

In [5]:
duckdb.sql("INSTALL spatial; LOAD spatial")

In [6]:
duckdb.sql("INSTALL h3 FROM community; LOAD h3;")

---

## 2. Municipality

Population data for Dutch municipalities is retrieved from the *Areas in the Netherlands 2025* dataset (CBS Table 86059NED), published by Statistics Netherlands (CBS). The dataset provides administrative and selected demographic information for municipalities, including the population count as of 1 January 2025.

The resulting table, `municipality_population`, holds one row per municipality with `municipality_code`, `municipality_name`, parent `province_code` / `province_name`, `population`, `code_degree_of_urbanisation`, and `degree_of_urbanisation`. The CBS degree-of-urbanisation classification is rendered in Dutch in the API response (`Niet stedelijk`, `Weinig stedelijk`, etc.); the integer code (`Code_54`) is mapped to the English labels used in the rest of the pipeline.

In [6]:
# Retrieve municipality-level population data from the CBS Open Data API.
# The query selects province and municipality identifiers, names, and population counts.
url = "https://opendata.cbs.nl/ODataApi/OData/86059NED/TypedDataSet?$select=Code_28,Naam_29,Code_1,Naam_2,Inwonertal_56,Code_54,Omschrijving_55"
response = requests.get(url).json()

# Convert API JSON response into a pandas DataFrame.
municipality_population = pd.DataFrame(response['value'])


# Rename CBS variable codes to descriptive column names.
# Variable definitions can be found in the dataset metadata:
# https://opendata.cbs.nl/ODataApi/OData/86059NED/DataProperties
municipality_population = municipality_population.rename(columns={'Code_28' : 'province_code', 
                                                                  'Naam_29' : 'province_name', 
                                                                  'Code_1' : 'municipality_code', 
                                                                  'Naam_2' : 'municipality_name',
                                                                  'Inwonertal_56' : 'population',
                                                                  'Code_54' : 'code_degree_of_urbanisation',
                                                                   'Omschrijving_55' : 'degree_of_urbanisation'})

# Remove trailing spaces from string columns --> it it necessary, it comes after renaming?
municipality_population = municipality_population.apply(lambda col: col.str.strip() if col.dtype == "string" else col)

# Translate text in degree of urbanization column 
mapping = {
    '1': "Very highly urbanized",
    '2': "Highly urbanized",
    '3': "Moderately urbanized",
    '4': "Slightly urbanized",
    '5': "Non-urbanized"
}

municipality_population['degree_of_urbanisation'] = (municipality_population['code_degree_of_urbanisation'].map(mapping))

## 3. Province

Because Dutch municipalities are nested within provinces, separate provincial population data is not retrieved. Provincial totals are derived by summing the municipality-level population counts from CBS Table 86059NED. The resulting table, `province_population`, holds one row per province with `province_code`, `province_name`, and `population`.

In [7]:
# Aggregate municipality populations to the provincial level.
province_population = municipality_population.groupby(['province_code', 'province_name'])['population'].sum().reset_index()

In [8]:
province_population

,province_code,province_name,population
0,PV20,Groningen,602833
1,PV21,Fryslân,664222
2,PV22,Drenthe,506529
3,PV23,Overijssel,1195789
4,PV24,Flevoland,456395
5,PV25,Gelderland,2161358
6,PV26,Utrecht,1409144
7,PV27,Noord-Holland,2992016
8,PV28,Zuid-Holland,3863397
9,PV29,Zeeland,392969


## 4. H3 grid

Population data for H3 cells is sourced from the [Kontur Population Dataset](https://data.humdata.org/dataset/kontur-population-netherlands), published on the Humanitarian Data Exchange (HDX) as *Kontur Population: Netherlands Population Density for 400m H3 Hexagons*. The dataset provides population estimates aggregated to H3 cells together with the corresponding cell geometries.

Although the dataset documentation refers to "400 m H3 hexagons" rather than an explicit H3 resolution, the H3 indexes were verified to correspond to resolution 8, consistent with the H3 grid generated in `01_boundaries`. In the preparation step, the H3 string indexes are converted to DuckDB's native H3 type (enabling H3 helper functions downstream), geometries are reprojected from EPSG:3857 (Web Mercator) to EPSG:4326 (WGS84), and the H3 resolution is attached as a validation column.

The resulting table, `h3_population`, holds one row per H3 cell with `h3_index`, `geom`, `population`, and `resolution`.

**Temporal mismatch.** The Kontur dataset reflects 2023 population estimates, whereas the CBS municipality and province populations (and the PDOK administrative boundaries) correspond to 2025. The impact on the analysis is expected to be limited because national-level population changes are gradual (CBS reports approximately 17.81 million in 2023, 17.94 million in 2024, and 18.4 million in 2025). Since H3-level population enters only via the MAUP robustness check in stage 08 and not the headline results, the residual gap is documented as a known limitation rather than corrected.

In [7]:
# Read the Kontur population dataset for the Netherlands.
# The dataset provides population estimates aggregated to H3 cells,
# together with the corresponding H3 cell geometries.
h3_population_arrow = duckdb.sql("""
WITH read_kontur AS (
    SELECT * RENAME(h3 as h3_index)
    FROM ST_Read('/vsigzip//vsicurl/https://geodata-eu-central-1-kontur-public.s3.amazonaws.com/kontur_datasets/kontur_population_NL_20231101.gpkg.gz')
)
SELECT * REPLACE(
    -- Convert H3 indexes from strings to DuckDB's native H3 type
    -- to enable use of H3 functions.
    h3_string_to_h3(h3_index) AS h3_index,

    -- Reproject geometries from Web Mercator (EPSG:3857) to WGS84 (EPSG:4326)
    -- for consistency with the administrative boundaries and H3 grid
    -- created in the 01_boundaries notebook.
    ST_Transform(geom, 'EPSG:3857', 'EPSG:4326', always_xy := true) AS geom
),
    -- Extract the H3 resolution associated with each cell.
    -- This serves as a validation step to confirm that the dataset
    -- is provided at resolution 8.
    h3_get_resolution(h3_index) AS resolution
    
FROM read_kontur
""").arrow()

# Convert arrow table into GeoDataFrame 
h3_population = gpd.GeoDataFrame.from_arrow(h3_population_arrow)

In [8]:
h3_population.head(5)

,fid,h3_index,population,geom,resolution
0,1,614309480223473663,741.0,"POLYGON ((-70.03617 12.58899, -70.03994 12.586...",8
1,2,614309480221376511,25.0,"POLYGON ((-70.02323 12.58265, -70.027 12.5804,...",8
2,3,614309480219279359,384.0,"POLYGON ((-70.0281 12.58913, -70.03186 12.5868...",8
3,4,614309480217182207,984.0,"POLYGON ((-70.03453 12.57588, -70.03829 12.573...",8
4,5,614309480215085055,573.0,"POLYGON ((-70.03939 12.58237, -70.04316 12.580...",8
